# Seed 42

# Eye Movement-Based Schizophrenia Recognition — Full Pipeline

| Cell | Mục đích |
|---|---|
| 1 | Mount Drive + cd vào project |
| 2 | 🔴 XÓA kết quả cũ (bỏ comment khi cần reset) |
| 3 | Install thư viện còn thiếu |
| 4 | Tier 1 — Preprocessing |
| 5 | Tier 2 — Feature Engineering |
| 6 | Tier 3 — Tabular (XGBoost) |
| 7 | Tier 4A — ResNet50 extraction |
| 8 | Tier 4B — Build graphs |
| 9 | Tier 4C — GNN-CEFAM training |
| 10 | Tier 4D — BiCA-HS training |
| 11 | Tier 5 — Meta-Learner |
| 12 | Tổng hợp kết quả |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition"

Mounted at /content/drive
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition


In [2]:
# 🔴 XÓA KẾT QUẢ CŨ — bỏ comment từng dòng tùy mức độ reset

# Xóa checkpoint + OOF Tier 4 (BẮT BUỘC sau khi fix C-1, C-2, C-3)
# !rm -rf results/bica/ results/cefam/ results/stgnn/ results/tier5/
# !rm -rf "Bidirectional Cross-Attention Hybrid Stream/results/checkpoints/"

# Xóa Tier 3
# !rm -rf results/baselines/

# Xóa graphs (rebuild từ đầu)
# !rm -rf data/processed/graphs/

# Xóa toàn bộ (chạy lại từ raw data)
!rm -rf results/ data/processed/ data/external/

In [4]:
# Colab đã có sẵn torch/sklearn/pandas — chỉ install thêm cái còn thiếu
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 25.0 MB/s eta 0:00:00


In [4]:
# Tier 1: Tạo category map + tiền xử lý dữ liệu thô
!python -m src.utils.generate_category_map
!python -m src.tier1_preprocessing.preprocess

Successfully generated category mapping for 100 images at data/metadata/stimulus_categories.csv
--- Tier 1: Loading raw data ---
Loading Fixations: 100% 160/160 [00:18<00:00,  8.66it/s]
Loading Fixations: 100% 48/48 [00:36<00:00,  1.31it/s]
Total raw fixations loaded: 293740
Spatial boundary filter: removed 5037 out of 293740 fixations (1.71%)
Temporal duration filter: removed 7666 out of 288703 fixations (2.66%)
Successfully saved 281037 fixations to data/processed/clean_fixations.parquet
--- Tier 1 Preprocessing Completed Successfully ---


In [5]:
# Tier 2: Trích xuất đặc trưng stimulus-level + subject-level delta
!python -m src.tier2_features.stimulus_features
!python -m src.tier2_features.subject_aggregator

Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Extracting features per trial...
100% 20695/20695 [00:23<00:00, 885.20it/s]
Extracted features for 20695 trials. Saved to data/processed/features_stimulus_level.csv
Loading stimulus-level features from data/processed/features_stimulus_level.csv...
Loading category mapping from data/metadata/stimulus_categories.csv...
Computing mean feature values per subject, per category...
Computing contextual delta features...
Aggregated subject-level features for 208 subjects. Saved to data/processed/features_subject_level.csv


In [6]:
# Tier 3: Tabular baseline (XGBoost)
!python -m src.tier3_tabular.tabular_models --model xgboost

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:21:29] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iter

In [2]:
!python -m src.tier3_tabular.tabular_models --model lightgbm

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.7615
  Stimulus-Level Auc: 0.8385
  Stimulus-Level F1: 0.7692
  Stimulus-Level Precision: 0.7423
  St

In [3]:
!python -m src.tier3_tabular.tabular_models --model catboost

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.8114
  Stimulus-Level Auc: 0.8971
  Stimulus-Level F1: 0.8057
  Stimulus-Level Precision: 0.8272
  St

In [7]:
# Tier 4A: Trích xuất ResNet50 visual features (2048-dim)
# Output: data/external/feature_dict_ResNet50.npy
!python -m src.utils.extract_resnet_features

 VISUAL FEATURE EXTRACTION (ResNet50 Baseline)
Found 100 stimulus images in EMS/Images.
Loading pre-trained ResNet50 on device: cuda...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 242MB/s]
Extracting feature maps for all stimulus images...
Extracting image features: 100% 100/100 [01:12<00:00,  1.38it/s]
Mapping visual features to subject fixations...
Mapping to trials: 100% 16716/16716 [00:56<00:00, 295.45it/s]

Successfully extracted ResNet50 features. Saved to data/external/feature_dict_ResNet50.npy
Total trials mapped: 16716
Feature vector dimension: 2048


In [8]:
# Tier 4B: Xây dựng đồ thị PyG từ fixation data + ResNet50 features
# Output: data/processed/graphs/graphs.pt
!python -m src.tier4_advanced.graph_builder

Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Global pupil stats (minor leakage — see NOTE above): Mean=1218.29, Std=614.17
Loading ResNet50 features from data/external/feature_dict_ResNet50.npy...
Detected visual feature dimension from dataset: 2048
Building spatiotemporal graphs...
100% 16716/16716 [00:37<00:00, 448.59it/s]
Successfully constructed and saved 16716 graphs at data/processed/graphs/graphs.pt


In [2]:
# Tier 4C: Huấn luyện GNN-CEFAM (4-fold GroupKFold)
# Output: results/cefam/cefam_oof_subject_preds.csv
!python scripts/train_tier4.py

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Found existing checkpoint at results/cefam/checkpoints/cefam_fold_0_best.pt. Evaluating...
Loaded checkpoint — Val Subject AUC: 0.9722

==================== 

In [3]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Found existing checkpoint at results/stgnn/checkpoints/stgnn_fold_0_best.pt. Evaluating...
Loaded checkpoint — Val Subject AUC: 0.9141

==================== Training Fold 1 ====================
Fold 1 training pupil stats: Mean=1256.3683, Std=633.2075
Train graphs: 11708, Val graphs: 3890
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.p

In [4]:
!python scripts/calibrate_threshold.py --preds results/stgnn/stgnn_subject_val_predictions.csv

Loading predictions from results/stgnn/stgnn_subject_val_predictions.csv...
Overall Subject-Level AUC: 0.9309

--- Metrics at Default Threshold (0.5000) ---
  Accuracy: 0.7812
  F1-Score: 0.7368
  Precision: 0.9245
  Recall: 0.6125

--- Optimized Metrics at Calibrated Threshold (0.4126) ---
  Accuracy: 0.8625
  F1-Score: 0.8675
  Precision: 0.8372
  Recall (Sensitivity): 0.9000
  Specificity: 0.8250


In [5]:
# Tier 4D: Huấn luyện BiCA-HS (4-fold GroupKFold)
# PYTHONPATH=. bắt buộc để import src.*
# Output: results/bica/bica_subject_val_predictions.csv
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --config configs/bica_config.yaml

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Found existing checkpoint at results/bica/checkpoints/bica_fold_0_best.pt. Resuming and evaluating...
Loaded checkpoint - Val Loss: 0.0595 | Val Trial AUC: 0.8669 | Val Subject AUC: 0.9141

==================== Training Fold 1 ====================
Train trials: 11708, Val trials: 3890
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/s

In [4]:
# Tier 5: Meta-Learner kết hợp Tier 3 + Tier 4
# Mặc định dùng XGBoost OOF + CEFAM OOF
!python scripts/run_tier5.py --plot --calibrate

2026-06-27 13:20:34.677712: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 13:20:34.745838: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.003, 0.989]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9405  ACC:

In [2]:
# Tier 5 (phiên bản dùng BiCA-HS OOF thay vì CEFAM)
!python scripts/run_tier5.py \
    --tier4-oof results/bica/bica_subject_val_predictions.csv \
    --plot --calibrate

2026-06-27 13:19:48.327982: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 13:19:48.400311: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.007, 0.889]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9522  ACC:

In [2]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s42/catboost_oof_subject_preds.csv \
    --tier4-oof results/bica_s42/bica_subject_val_predictions.csv \
    --output-dir results/tier5_s42/ \
    --calibrate

2026-06-27 16:36:50.892864: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.007, 0.889]
[Tier5] Test prediction files not found; skipping test inference.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8981  ACC: 0.8125  Brier: 0.1480  BSS: 0.4082  ECE: 0.1508

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9522  ACC: 0.9062  Brier: 0.1215  BSS: 0.5141  ECE: 0.1859

=== Tier 5 Meta-Learner Training ===
Input shape: (160, 2)  |  Class balance: 80/80 (SZ/HC)
[WeightedAvg] Optuna not installed; using uniform weights.
[LogisticReg] CV AUC: 0.9537 ± 0.0166
[LogisticReg] Coefficients: β_tab=2.3603  β_bica=3.7497  bias=-2.8955



In [13]:
# Tổng hợp kết quả
import json, os
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

def show(path, name):
    if not os.path.exists(path):
        return
    df = pd.read_csv(path)
    if 'Label' not in df.columns or 'Pred_Proba_Subject' not in df.columns:
        return
    y, p = df['Label'].values, df['Pred_Proba_Subject'].values
    print(f"{name:<35} AUC={roc_auc_score(y,p):.4f}  ACC={accuracy_score(y,(p>=.5).astype(int)):.4f}  F1={f1_score(y,(p>=.5).astype(int)):.4f}")

print("=" * 70)
show("results/baselines/xgboost_oof_subject_preds.csv",  "Tier 3 XGBoost")
show("results/cefam/cefam_oof_subject_preds.csv",        "Tier 4 GNN-CEFAM")
show("results/bica/bica_subject_val_predictions.csv",    "Tier 4 BiCA-HS")

t5 = "results/tier5/tier5_results_summary.json"
if os.path.exists(t5):
    d = json.load(open(t5))
    if "best_model" in d:
        b = d["best_model"]
        print(f"{'Tier 5 Meta-Learner':<35} AUC={b.get('auc',0):.4f}  ACC={b.get('acc',0):.4f}  F1={b.get('f1',0):.4f}")
print("=" * 70)
print("SOTA (MSNet, IEEE TNNLS 2025):      AUC=0.8854  ACC=0.8125")

Tier 3 XGBoost                      AUC=0.8702  ACC=0.7812  F1=0.7853
Tier 4 GNN-CEFAM                    AUC=0.9405  ACC=0.8688  F1=0.8679
Tier 4 BiCA-HS                      AUC=0.9522  ACC=0.9062  F1=0.9123
SOTA (MSNet, IEEE TNNLS 2025):      AUC=0.8854  ACC=0.8125


In [5]:
!rm -rf experiments/ablation/results/ experiments/ablation/figures/

!python experiments/ablation/run_ablation_analysis.py

!python experiments/ablation/plot_ablation.py

ABLATION STUDY - Eye Movement-Based Schizophrenia Recognition

Loading model predictions...
Loaded 4 models: ['GNN+CEFAM (Full Hybrid)', 'ST-GNN (GNN Only)', 'BiCA-HS (Transformer)', 'XGBoost (Tabular Only)']

F1: FULL MODEL COMPARISON (Main Ablation Table)

--- GNN+CEFAM (Full Hybrid) ---
  AUC-ROC:  0.9405
  ACC @0.5: 0.8688
  F1  @0.5: 0.8679
  ACC @opt: 0.8750 (th=0.3026)
  F1  @opt: 0.8810
  Sens@opt: 0.9250
  Spec@opt: 0.8250

--- ST-GNN (GNN Only) ---
  AUC-ROC:  0.9309
  ACC @0.5: 0.7812
  F1  @0.5: 0.7368
  ACC @opt: 0.8625 (th=0.4126)
  F1  @opt: 0.8675
  Sens@opt: 0.9000
  Spec@opt: 0.8250

--- BiCA-HS (Transformer) ---
  AUC-ROC:  0.9522
  ACC @0.5: 0.9062
  F1  @0.5: 0.9123
  ACC @opt: 0.9062 (th=0.4990)
  F1  @opt: 0.9123
  Sens@opt: 0.9750
  Spec@opt: 0.8375

--- XGBoost (Tabular Only) ---
  AUC-ROC:  0.8702
  ACC @0.5: 0.7812
  F1  @0.5: 0.7853
  ACC @opt: 0.8063 (th=0.5953)
  F1  @opt: 0.8050
  Sens@opt: 0.8000
  Spec@opt: 0.8125

F2: COMPONENT CONTRIBUTION ANALYSIS

 

In [6]:
!python scratch_delong_test.py

 DELONG SIGNIFICANCE TEST: BiCA-HS vs Tier 5 Ensembles
Number of subjects : 160
Tier 4 (BiCA-HS) AUC : 0.9522
------------------------------------------------------------
 1. CROSS-VALIDATED ENSEMBLE (Generalization performance)
------------------------------------------------------------
Tier 5 (CV Meta) AUC : 0.9437
Z-statistic          : 0.6074
P-value              : 0.543587
Result is NOT statistically significant at alpha=0.05 (p >= 0.05).
------------------------------------------------------------
 2. FULL META-LEARNER ENSEMBLE (Final model deployment)
------------------------------------------------------------
Tier 5 (Full Meta) AUC: 0.9597
Z-statistic           : -0.7482
P-value               : 0.454346
Result is NOT statistically significant at alpha=0.05 (p >= 0.05).
The difference in performance could be due to chance.


In [7]:
!python scratch_feature_importance.py

  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:25:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:25:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, i

# Seed 123

In [4]:
!python -m src.tier3_tabular.tabular_models --model xgboost --seed 123
!python -m src.tier3_tabular.tabular_models --model catboost --seed 123

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:42:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iter

In [5]:
!python -m src.tier3_tabular.tabular_models --model lightgbm --seed 123

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.7866
  Stimulus-Level Auc: 0.8430
  Stimulus-Level F1: 0.7935
  Stimulus-Level Precision: 0.7658
  St

In [2]:
!python scripts/train_tier4.py --seed 123

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0437 TrAUC=0.9453 | VaLoss=0.0807 VaTrialAUC=0.8590 VaSubjAUC=0.8838
Ep 005/200 | TrLoss=0.0090 TrAUC=0.9971 | VaLoss=0.0827 VaTrialAUC

In [6]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml --seed 123

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0624 TrAUC=0.8117 | VaLoss=0.0687 VaTrialAUC=0.7940 VaSubjAUC=0.8359
Ep 005/200 | TrLoss=0.0461 TrAUC=0.8845 | VaLoss=0.0569 VaTrialAUC=0.8111 VaSubjAUC=0.8788
Ep 010/200 | TrLoss=0.0427 TrAUC=0.9024 | VaLoss=0.0683 VaTrialAUC=0.8022 VaSubjAUC=0.8359
Ep 015/200 | TrLoss=0.0398 TrAUC=0.9162 | VaLoss=0.0678 VaTrialAUC=0.8261 VaSubjAUC=0.8687
Ep 020/200 | TrLoss=0.038

In [3]:
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" \
    --config configs/bica_config.yaml --seed 123

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0624 | Train AUC: 0.8908 | Val Loss: 0.0922 | Val Trial AUC: 0.7297 | Val Subject AUC: 0.7374
Epoch 005/150 | Train Loss: 0.0436 | Train AUC: 0.9268 | Val Loss: 0.0677 | Val Trial AUC: 0.8406 | Val Subject AUC: 0.8864
Epoch 010/150 | Train Loss: 0.0423 | Train AUC: 0.9315 | Val Loss: 0.0872 | Val Trial AUC: 0.8223 | Val Sub

In [2]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s123/catboost_oof_subject_preds.csv \
    --tier4-oof results/bica_s123/bica_subject_val_predictions.csv \
    --output-dir results/tier5_s123/ \
    --calibrate

2026-06-27 16:06:55.306611: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 16:06:55.377297: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.000, 0.895]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.9006  ACC: 0.8187  Brier: 0.1503  BSS: 0.3989  ECE: 0.1316

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9314  ACC:

# Seed 456

In [7]:
!python -m src.tier3_tabular.tabular_models --model xgboost --seed 456
!python -m src.tier3_tabular.tabular_models --model catboost --seed 456

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:49:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iter

In [7]:
!python -m src.tier3_tabular.tabular_models --model lightgbm --seed 456

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.7803
  Stimulus-Level Auc: 0.8453
  Stimulus-Level F1: 0.7835
  Stimulus-Level Precision: 0.7694
  St

In [5]:
!python scripts/train_tier4.py --seed 456

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0426 TrAUC=0.9495 | VaLoss=0.0784 VaTrialAUC=0.8720 VaSubjAUC=0.8914
Ep 005/200 | TrLoss=0.0078 TrAUC=0.9981 | VaLoss=0.1463 VaTrialAUC

In [8]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml --seed 456

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0644 TrAUC=0.7977 | VaLoss=0.0635 VaTrialAUC=0.8018 VaSubjAUC=0.8510
Ep 005/200 | TrLoss=0.0445 TrAUC=0.8943 | VaLoss=0.0749 VaTrialAUC=0.8122 VaSubjAUC=0.8586
Ep 010/200 | TrLoss=0.0420 TrAUC=0.9075 | VaLoss=0.0744 VaTrialAUC=0.8268 VaSubjAUC=0.8737
Ep 015/200 | TrLoss=0.0387 TrAUC=0.9219 | VaLoss=0.0659 VaTrialAUC=0.8304 VaSubjAUC=0.8763
Ep 020/200 | TrLoss=0.036

In [6]:
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" \
    --config configs/bica_config.yaml --seed 456

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0604 | Train AUC: 0.8969 | Val Loss: 0.0737 | Val Trial AUC: 0.8146 | Val Subject AUC: 0.8460
Epoch 005/150 | Train Loss: 0.0431 | Train AUC: 0.9294 | Val Loss: 0.0647 | Val Trial AUC: 0.8425 | Val Subject AUC: 0.8813
Epoch 010/150 | Train Loss: 0.0409 | Train AUC: 0.9356 | Val Loss: 0.0883 | Val Trial AUC: 0.8389 | Val Sub

In [3]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s456/catboost_oof_subject_preds.csv \
    --tier4-oof results/bica_s456/bica_subject_val_predictions.csv \
    --output-dir results/tier5_s456/ \
    --calibrate

2026-06-27 16:07:18.761141: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 16:07:18.828556: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.009, 0.901]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8981  ACC: 0.8000  Brier: 0.1461  BSS: 0.4157  ECE: 0.1476

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9527  ACC:

In [9]:
!python scripts/aggregate_multiseed.py

Model                                   AUC              ACC               F1             Sens
XGBoost                       0.8716±0.0046    0.7771±0.0106    0.7765±0.0120    0.7750±0.0204
CatBoost                      0.8990±0.0012    0.8104±0.0078    0.8052±0.0076    0.7833±0.0059
GNN-CEFAM                     0.9398±0.0047    0.8667±0.0179    0.8629±0.0164    0.8375±0.0177
BiCA-HS                       0.9454±0.0099    0.8792±0.0262    0.8887±0.0232    0.9625±0.0177
Tier5 (CatBoost+BiCA)         0.9628±0.0024    0.8750±0.0102    0.8728±0.0111    0.8583±0.0156
SOTA (MSNet 2025)                    0.8854           0.8125

--- Per-seed AUC ---
  XGBoost                   : s42=0.8702, s123=0.8667, s456=0.8778
  CatBoost                  : s42=0.8981, s123=0.9006, s456=0.8981
  GNN-CEFAM                 : s42=0.9405, s123=0.9453, s456=0.9337
  BiCA-HS                   : s42=0.9522, s123=0.9314, s456=0.9527
  Tier5 (CatBoost+BiCA)     : s42=0.9594, s123=0.9639, s456=0.9650


In [10]:
!python scripts/bootstrap_ci_vs_sota.py

Bootstrap 95% CI (n=10,000) — vs SOTA AUC = 0.8854

BiCA-HS
    Seed |     AUC |             95% CI |   p-value | Sig?
  -------+---------+--------------------+-----------+------
      42 | 0.9522 | [0.9170, 0.9801] |    0.0002 | YES
     123 | 0.9314 | [0.8905, 0.9658] |    0.0162 | YES
     456 | 0.9527 | [0.9151, 0.9833] |    0.0011 | YES
    POOL |        | [0.9012, 0.9797] |    0.0058 | YES

GNN-CEFAM
    Seed |     AUC |             95% CI |   p-value | Sig?
  -------+---------+--------------------+-----------+------
      42 | 0.9405 | [0.9020, 0.9715] |    0.0039 | YES
     123 | 0.9453 | [0.9090, 0.9733] |    0.0010 | YES
     456 | 0.9337 | [0.8931, 0.9675] |    0.0120 | YES
    POOL |        | [0.8997, 0.9712] |    0.0056 | YES

CatBoost
    Seed |     AUC |             95% CI |   p-value | Sig?
  -------+---------+--------------------+-----------+------
      42 | 0.8981 | [0.8444, 0.9433] |    0.2968 | NO
     123 | 0.9006 | [0.8492, 0.9436] |    0.2611 | NO
     456 | 0.8

In [11]:
!rm -rf experiments/ablation/results/ experiments/ablation/figures/

!python experiments/ablation/run_ablation_analysis.py

!python experiments/ablation/plot_ablation.py

ABLATION STUDY - Eye Movement-Based Schizophrenia Recognition

Loading model predictions...
Loaded 4 models: ['GNN+CEFAM (Full Hybrid)', 'ST-GNN (GNN Only)', 'BiCA-HS (Transformer)', 'XGBoost (Tabular Only)']

F1: FULL MODEL COMPARISON (Main Ablation Table)

--- GNN+CEFAM (Full Hybrid) ---
  AUC-ROC:  0.9405
  ACC @0.5: 0.8688
  F1  @0.5: 0.8679
  ACC @opt: 0.8750 (th=0.3026)
  F1  @opt: 0.8810
  Sens@opt: 0.9250
  Spec@opt: 0.8250

--- ST-GNN (GNN Only) ---
  AUC-ROC:  0.9309
  ACC @0.5: 0.7812
  F1  @0.5: 0.7368
  ACC @opt: 0.8625 (th=0.4126)
  F1  @opt: 0.8675
  Sens@opt: 0.9000
  Spec@opt: 0.8250

--- BiCA-HS (Transformer) ---
  AUC-ROC:  0.9522
  ACC @0.5: 0.9062
  F1  @0.5: 0.9123
  ACC @opt: 0.9062 (th=0.4990)
  F1  @opt: 0.9123
  Sens@opt: 0.9750
  Spec@opt: 0.8375

--- XGBoost (Tabular Only) ---
  AUC-ROC:  0.8702
  ACC @0.5: 0.7812
  F1  @0.5: 0.7853
  ACC @opt: 0.8063 (th=0.5953)
  F1  @opt: 0.8050
  Sens@opt: 0.8000
  Spec@opt: 0.8125

F2: COMPONENT CONTRIBUTION ANALYSIS

 